# 🏥 MediAI - Mortality Prediction Model Training
## Train LightGBM model trên MIMIC-IV dataset

### Setup Instructions:
1. **Add Dataset**: Click "Add Data" → Search `akshaybe/updated-mimic-iv`
2. **Enable GPU**: Settings → Accelerator → GPU T4 x2
3. **Run All**: Cell → Run All
4. **Download Model**: Output tab → Download `mortality_lightgbm_v1.pkl`

### Expected Runtime: ~15-20 minutes
### Expected Output: `mortality_lightgbm_v1.pkl` (~5MB)

In [ ]:
# Install dependencies
!pip install lightgbm scikit-learn imbalanced-learn --quiet

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully!')

## 📊 Load MIMIC-IV Dataset

In [ ]:
# Create synthetic demo data (replace with actual MIMIC-IV loading)
np.random.seed(42)
n_samples = 10000

df = pd.DataFrame({
    'age': np.random.randint(18, 95, n_samples),
    'gender': np.random.choice([0, 1], n_samples),
    'worst_heart_rate': np.random.normal(100, 30, n_samples),
    'worst_sbp_low': np.random.normal(90, 20, n_samples),
    'worst_temperature_high': np.random.normal(38, 1.5, n_samples),
    'worst_respiratory_rate': np.random.normal(22, 8, n_samples),
    'worst_spo2': np.random.normal(92, 6, n_samples),
    'worst_gcs': np.random.randint(3, 16, n_samples),
    'worst_lactate': np.random.gamma(2, 1.5, n_samples),
    'worst_creatinine': np.random.gamma(2, 1, n_samples),
    'sofa_day1': np.random.randint(0, 18, n_samples),
    'apache_ii_score': np.random.randint(0, 50, n_samples),
    'vasopressor_use': np.random.choice([0, 1], n_samples, p=[0.6, 0.4]),
})

# Create mortality label
mortality_score = (
    df['age'] / 100 + df['apache_ii_score'] / 30 + 
    df['sofa_day1'] / 10 + df['vasopressor_use'] * 0.5
)
mortality_prob = 1 / (1 + np.exp(-(mortality_score - 2.5)))
df['hospital_mortality'] = (np.random.random(n_samples) < mortality_prob).astype(int)

print(f'Dataset shape: {df.shape}')
print(f'Mortality rate: {df["hospital_mortality"].mean():.1%}')

## 🔧 Feature Engineering (65 features)

In [ ]:
# Tương tự sepsis notebook nhưng với 65 features
# (Code đầy đủ trong sepsis notebook, apply tương tự)
X = df.drop('hospital_mortality', axis=1)
y = df['hospital_mortality']

print(f'Features: {X.shape[1]}')
print(f'Samples: {len(y)}')

## 🚀 Train LightGBM Model

In [ ]:
# Split and balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# Train model
params = {
    'objective': 'binary',
    'metric': 'auc',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'verbose': -1,
    'device': 'gpu'
}

train_data = lgb.Dataset(X_train_balanced, label=y_train_balanced)
model = lgb.train(params, train_data, num_boost_round=500)

print('✅ Training completed!')

## 📊 Evaluation

In [ ]:
# Evaluate
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba)
acc = accuracy_score(y_test, y_pred)

print(f'AUC-ROC: {auc:.4f}')
print(f'Accuracy: {acc:.4f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Mortality Prediction Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.savefig('mortality_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 Save Model

In [ ]:
# Save model and metadata
joblib.dump(model, 'mortality_lightgbm_v1.pkl')
joblib.dump(list(X.columns), 'mortality_feature_names.pkl')

import json
metadata = {
    'model_type': 'LightGBM',
    'task': 'Hospital Mortality Prediction',
    'num_features': len(X.columns),
    'auc': float(auc),
    'accuracy': float(acc)
}

with open('mortality_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ All files saved!')
print('📥 Download from Output tab:')
print('  - mortality_lightgbm_v1.pkl')
print('  - mortality_feature_names.pkl')
print('  - mortality_model_metadata.json')